# 03 · Watchers, evaluators & launching a full experiment

Once `.env` / `braintrust.env` are configured, this notebook shows the complete
experiment lifecycle used by the production scripts:

1. **Preflight** — validate prompt + dataset with zero model credits.
2. **Evaluators** — the three Braintrust scorers registered by the runner.
3. **Launch** — start a full experiment run (shown as a print-only preview).
4. **Watchers** — monitor progress from a completed run's JSONL manifest.
5. **Report** — the post-run scoring/reporting commands (print-only).


## 0. Bootstrap: repo path + credentials

In [1]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "constants.py").exists()
)
sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)


Repo root: /Users/morningstar/Desktop/Cold_Storage/AMFAM_capstone


In [2]:
from src.braintrust_config import load_braintrust_config
from src.env_utils import require_env

config = load_braintrust_config()      # braintrust.env first, then .env
api_key = require_env("OPENROUTER_API_KEY")[0]

print("project:", config.project_name)
print("project_id:", config.project_id)
print("dataset:", config.dataset_project, "/", config.dataset)
print("model:", config.model)
print("braintrust api_key set:", bool(config.api_key))
print("openrouter api_key set:", bool(api_key))


project: AMFAM v2
project_id: ba0346b3-cad8-463d-b758-afddafd9f0d0
dataset: AMFAM v2 / fixed_size_sampled
model: qwen/qwen3.7-flash
braintrust api_key set: True
openrouter api_key set: True


## 1. Preflight (zero credits)

`preflight_eval.py` checks that the prompt version resolves and the dataset is
reachable under the current credentials **without sending any model request**.
Run it before any eval to catch setup problems early.

In [3]:
import subprocess
import sys


def run_script(rel_script: str, *args: str) -> None:
    cmd = [sys.executable, str(ROOT / "scripts" / rel_script), *args]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)


run_script("braintrust/preflight_eval.py", "--dataset", config.dataset, "--prompt-version", "v17.2")


$ /Library/Developer/CommandLineTools/usr/bin/python3 /Users/morningstar/Desktop/Cold_Storage/AMFAM_capstone/scripts/braintrust/preflight_eval.py --dataset fixed_size_sampled --prompt-version v17.2


/Users/morningstar/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


PREFLIGHT OK: prompt=v17.2 dataset=AMFAM v2/fixed_size_sampled rows=160


## 2. Evaluators registered by the runner

The eval runner wraps the OpenAI client with `braintrust.wrap_openai()` and runs
`braintrust.Eval(..., scores=[...])`. Exactly three scorers are registered:

- **`exact_match`** — `output.strip().lower() == expected_class`, scored 1.0/0.0.
- **`failure`** — rows whose output starts with `ERROR: ` (count as misses too).
- **`cost`** — each row's actual billed USD from OpenRouter's `usage.cost`.

Near-miss (runner-up == expected while predicted != expected) is **not** a Braintrust
scorer — it is computed locally from the runner-up line the manifest records, by
`score_manifest.py`.

Abridged registration from `braintrust_openrouter_input.py`:

```python
from braintrust import Eval, wrap_openai
from openai import OpenAI

client = wrap_openai(OpenAI(base_url=OPENROUTER_BASE_URL, api_key=key))

Eval(
    dataset=dataset_rows,
    task=classify_row,          # returns (output, metadata) per row
    scores=[exact_match, failure, cost],
    metadata={"model": model, "prompt_version": prompt_version},
    max_concurrency=8,
)
```


## 3. Launch a full experiment (preview only)

Launching the runner against the configured dataset writes every completed row to a
**local JSONL manifest** (the durable checkpoint) as well as Braintrust, so the run
survives crashes, Braintrust limits, and quota errors. A full 160-image run spends
OpenRouter credits (~$0.0004/image), so this demo only prints the launch command.

In [4]:
experiment_name = f"notebook_full_{config.model.replace('/', '_')}_v17.2"
manifest = ROOT / "reports" / "manifests" / f"{experiment_name}.jsonl"

cmd = [
    sys.executable,
    str(ROOT / "scripts" / "braintrust" / "braintrust_openrouter_input.py"),
    "--dataset", config.dataset,
    "--prompt-version", "v17.2",
    "--model", config.model,
    "--experiment-name", experiment_name,
    "--manifest", str(manifest),
]
print("$", " ".join(cmd))
print("\n(Not launched: a full 160-image run would spend OpenRouter credits.)")


$ /Library/Developer/CommandLineTools/usr/bin/python3 /Users/morningstar/Desktop/Cold_Storage/AMFAM_capstone/scripts/braintrust/braintrust_openrouter_input.py --dataset fixed_size_sampled --prompt-version v17.2 --model qwen/qwen3.7-flash --experiment-name notebook_full_qwen_qwen3.7-flash_v17.2 --manifest /Users/morningstar/Desktop/Cold_Storage/AMFAM_capstone/reports/manifests/notebook_full_qwen_qwen3.7-flash_v17.2.jsonl

(Not launched: a full 160-image run would spend OpenRouter credits.)


## 4. Watch a run from the manifest

Each manifest record carries a `status` (`completed` / `error` / `empty`) plus
`runner_up` and `cost`; the eval runner retries transient provider failures up to
`MAX_TRIES=3`, growing `max_tokens` toward `MAX_TOKENS_CAP=32768` on length caps.
Here we count statuses from a **completed** run's manifest shipped with the repo
(`reports/manifests/eval_v17_v1.jsonl`) — read-only.

In [5]:
import json

from pathlib import Path


def manifest_status_counts(path: Path) -> dict:
    counts: dict[str, int] = {}
    if not path.exists():
        return {"(manifest not created yet)": 0}
    for line in path.read_text().splitlines()[1:]:
        if not line.strip():
            continue
        rec = json.loads(line)
        status = rec.get("status", "empty")
        counts[status] = counts.get(status, 0) + 1
    return counts


# A completed run's manifest, shipped with the repo (read-only demo).
manifest = ROOT / "reports" / "manifests" / "eval_v17_v1.jsonl"
print("Watching:", manifest.name)
manifest_status_counts(manifest)


Watching: eval_v17_v1.jsonl


{'completed': 159, 'error': 1}

## 5. Crash-proof resume

If a run dies (crash, Braintrust cap, quota 403), re-invoke the runner through
`resume_until_complete.py` until `--expected-rows` unique filenames have a final
status. Completed rows are skipped; failed/error rows are re-attempted. On completion
it auto-scores the manifest locally with `score_manifest.py` (no Braintrust scorer
credits). For production, `run_eval_queue.py` chains multiple jobs sequentially with
preflight checks and manifest verification between jobs.

In [6]:
# Illustrative: re-invokes the runner until every row is finished.
# Expected rows must equal the dataset slice size (fixed_size_sampled = 160).
cmd = [
    sys.executable,
    str(ROOT / "scripts" / "braintrust" / "resume_until_complete.py"),
    "--dataset", config.dataset,
    "--prompt-version", "v17.2",
    "--model", config.model,
    "--max-tokens", "8192",
    "--experiment-name", "qwen3.7-flash_v17_v1",
    "--manifest", str(manifest),
    "--expected-rows", "160",
]
print("$", " ".join(cmd))
# subprocess.run(cmd, cwd=ROOT, check=True)   # uncomment to run


$ /Library/Developer/CommandLineTools/usr/bin/python3 /Users/morningstar/Desktop/Cold_Storage/AMFAM_capstone/scripts/braintrust/resume_until_complete.py --dataset fixed_size_sampled --prompt-version v17.2 --model qwen/qwen3.7-flash --max-tokens 8192 --experiment-name qwen3.7-flash_v17_v1 --manifest /Users/morningstar/Desktop/Cold_Storage/AMFAM_capstone/reports/manifests/eval_v17_v1.jsonl --expected-rows 160


## 6. Post-run scoring & reporting (preview only)

The full reporting chain (also wired in `scripts/braintrust/`) against the completed
`qwen3.7-flash_v17_v1` manifest above:

1. `score_manifest.py` — local scoring from the manifest, no Braintrust credits.
2. `summarize_braintrust_experiment.py` — per-image OK/MISS summary + exact_match.
3. `braintrust_report.py` — accuracy, confusion matrix (PNG+MD), misclassification
   reasoning, cost breakdown (adjust `--input-price`/`--output-price` to the current
   OpenRouter model rates).
4. `braintrust_metrics_visual.py` — per-class chart + heatmap, and appends the
   experiment to `docs/experiments/experiment_log.md`.

These scripts write files into `reports/` and append to `docs/experiments/experiment_log.md`,
so this demo prints the commands instead of running them.

In [7]:
print("$ python scripts/braintrust/score_manifest.py --manifest", manifest)
print("$ python scripts/braintrust/summarize_braintrust_experiment.py --experiment qwen3.7-flash_v17_v1")
print("$ python scripts/braintrust/braintrust_report.py \\")
print(f"      --experiment qwen3.7-flash_v17_v1 --model {config.model} --prompt-version v17.2 \\")
print(f"      --dataset {config.dataset} --images-per-class 10 \\")
print("      --input-price 0.03 --output-price 0.13")
print("$ python scripts/braintrust/braintrust_metrics_visual.py qwen3.7-flash_v17_v1")


$ python scripts/braintrust/score_manifest.py --manifest /Users/morningstar/Desktop/Cold_Storage/AMFAM_capstone/reports/manifests/eval_v17_v1.jsonl
$ python scripts/braintrust/summarize_braintrust_experiment.py --experiment qwen3.7-flash_v17_v1
$ python scripts/braintrust/braintrust_report.py \
      --experiment qwen3.7-flash_v17_v1 --model qwen/qwen3.7-flash --prompt-version v17.2 \
      --dataset fixed_size_sampled --images-per-class 10 \
      --input-price 0.03 --output-price 0.13
$ python scripts/braintrust/braintrust_metrics_visual.py qwen3.7-flash_v17_v1


## Recap

1. Preflight validates prompt + dataset with zero credits.
2. The runner registers `exact_match`, `failure`, and `cost` scorers.
3. A full run writes every row to the local manifest as well as Braintrust.
4. Watch progress from the manifest; resume with `resume_until_complete.py`.
5. Score locally, then generate the summary / report / charts / experiment log.

Inspect individual row traces in the Braintrust UI (each span carries `raw_response`,
`reasoning`, `model`, `prompt_version`, `filename`, and error rows add `error`/`attempts`).